# HumanEval dataset exploration

Inspect archived Llama 3 8B generations, paired canonical solutions, and diff-based token labels.
Split by task so variants of the same problem stay together.

In [1]:
from pathlib import Path
import sys

if Path("traces.py").is_file():
    sys.path.insert(0, str(Path.cwd().parent))

In [2]:
import pandas as pd
from IPython.display import display

from experiments.data import PILOT_LIMITS, load_examples, split_examples
from experiments.data import summarize

benchmark = "humaneval"
examples = load_examples(benchmark)
full_splits = split_examples(examples)
pilot_splits = split_examples(examples, PILOT_LIMITS)

display(pd.DataFrame(summarize(full_splits)).assign(profile="full"))
display(pd.DataFrame(summarize(pilot_splits)).assign(profile="pilot"))

,split,examples,tasks,defective_examples,archived_hallucinations,clean_tokens,defective_tokens,ignored_tokens,profile
0,train,138,69,58,69,7206,1169,20926,full
1,validation,46,23,19,23,2245,741,7252,full
2,test,46,23,19,23,3049,665,8148,full


,split,examples,tasks,defective_examples,archived_hallucinations,clean_tokens,defective_tokens,ignored_tokens,profile
0,train,48,40,20,26,2372,290,7034,pilot
1,validation,16,13,7,8,869,84,2432,pilot
2,test,24,19,10,10,1579,549,4209,pilot


In [3]:
for label in (0, 1):
    example = next(row for row in full_splits["train"] if row.defect == label)
    print(f"Task {example.task_id} | annotated defect = {example.defect}")
    print(example.question[:500])
    print()
    print(example.reply[:600])
    print()

Task 145 | annotated defect = 0
Write a function which sorts the given list of integers
in ascending order according to the sum of their digits.
Note: if there are several items with similar sum of their digits,
order them based on their index in original list.
For example:
>>> order_by_points([1, 11, -1, -11, -12]) == [-1, -11, 1, -12, 11]
>>> order_by_points([]) == []

Respond only with code. Start the response with "```python
def order_by_points(nums):" and complete it.


```python
def order_by_points(nums):
    def weight(x):
        x_list = list(str(x))
        if x_list[0] == "-":
            x_list = x_list[1:]
            x_list = list(map(int, x_list))
            x_list[0] = -x_list[0]
        else:
            x_list = list(map(int, x_list))
        return sum(x_list)

    return sorted(nums, key=weight)
```


Task 19 | annotated defect = 1
Input is a space-delimited string of numberals from 'zero' to 'nine'.
Valid choices are 'zero', 'one', 'two', 'three', 'four', 'five', 